## Evalution Metrics

### Descriptive

#### GPT-4o

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import norm, wasserstein_distance, kstest, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'gpt_predictions_random_full_4o_new.csv'   
METRICS_OUT_FILE  = 'final_metrics_summary_descrip_4onew_newcal_CI_99.csv'        
DETAILED_OUT_FILE = 'final_metrics_detailed_descrip_4onew_newcal_CI_99.csv'       
PLOT_DIR          = 'plots_random_4o_descri_4onew_newcal_CI_99'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

# Using a denser linspace for quantile granularity
Q_VALUES = np.linspace(0.01, 0.99, 1000)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    decision_a = np.array(decision_a)
    demands_xi = np.array(demands_xi)
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    return np.mean(q * underage + (1 - q) * overage)

def get_discrete_quantile(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return 0
    idx = int(np.ceil(n * q)) - 1
    return sorted_data[max(0, min(idx, n - 1))]

def get_normal_quantile(mu, sigma, q):
    if pd.isna(mu) or pd.isna(sigma): return 0
    sigma = max(sigma, 0.001)
    return max(0, norm.ppf(q, loc=mu, scale=sigma))

# ==========================================
# MAIN LOGIC
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        gpt_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    global_pool = np.sort(true_df[sales_cols].values.flatten()[~np.isnan(true_df[sales_cols].values.flatten())])
    global_mean = np.mean(global_pool)

    summary_results, detailed_results = [], []
    unique_items = gpt_df['article_id'].unique()

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        t_row = true_df[true_df['article_id'] == art_id]
        if t_row.empty: continue
        t_demands = np.sort(t_row[sales_cols].values.flatten()[~np.isnan(t_row[sales_cols].values.flatten())])
        t_mean = np.mean(t_demands)

        gpt_row = gpt_df[gpt_df['article_id'] == art_id].iloc[0]
        pred_mean, pred_std = gpt_row.get('pred_mean'), gpt_row.get('pred_std')
        if pd.isna(pred_mean) or pd.isna(pred_std): continue

        synth_samples = np.maximum(0, np.random.normal(pred_mean, max(pred_std, 0.001), size=5000))
        wass_gpt = wasserstein_distance(t_demands, synth_samples)
        wass_base = wasserstein_distance(t_demands, global_pool)
        ks_gpt, _ = kstest(t_demands, 'norm', args=(pred_mean, max(pred_std, 0.001)))
        ks_base, _ = ks_2samp(t_demands, global_pool)

        r_gpt, r_base = [], []
        for q in Q_VALUES:
            a_star = get_discrete_quantile(t_demands, q)
            a_gpt = get_normal_quantile(pred_mean, pred_std, q)
            a_base = get_discrete_quantile(global_pool, q)
            
            lo, lg, lb = [calculate_newsvendor_loss(a, t_demands, q) for a in [a_star, a_gpt, a_base]]
            rg, rb = [min(lo/x, 1.0) if x > 1e-9 else 1.0 for x in [lg, lb]]
            r_gpt.append(rg); r_base.append(rb)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 4), 'CR_gpt': rg, 'CR_base': rb})

        # Identify the specific quantile that yielded the Worst CR for this item
        worst_q_gpt = Q_VALUES[np.argmin(r_gpt)]
        worst_q_base = Q_VALUES[np.argmin(r_base)]

        summary_results.append({
            'article_id': art_id, 'method': 'Relevance Selection Method', 'true_mean': t_mean, 'pred_mean': pred_mean,
            'abs_error': abs(pred_mean - t_mean), 'squared_error': (pred_mean - t_mean)**2,
            'AverageCR': np.mean(r_gpt), 'WorstCR': np.min(r_gpt), 'WorstCR_Quantile': worst_q_gpt,
            'Wasserstein': wass_gpt, 'Kolmogorov': ks_gpt
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': t_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - t_mean), 'squared_error': (global_mean - t_mean)**2,
            'AverageCR': np.mean(r_base), 'WorstCR': np.min(r_base), 'WorstCR_Quantile': worst_q_base,
            'Wasserstein': wass_base, 'Kolmogorov': ks_base
        })

    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS & VISUALIZATION
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*85)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      ")
    print("="*85)

    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'Wasserstein', 'Kolmogorov']
    # Added 'median' to the aggregation
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_list = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        m_row = {'Method': method}
        print(f"{'Metric':15} | {'Mean':>8} | {'Median':>8} | {'95% CI Range':>22}")
        print("-" * 65)

        for m in metrics:
            avg, med, std, n = stats_agg.loc[method, (m, ['mean', 'median', 'std', 'count'])]
            margin = 1.96 * (std / np.sqrt(n))
            
            print(f"{m:15} | {avg:8.4f} | {med:8.4f} | [{avg-margin:7.4f}, {avg+margin:7.4f}]")
            m_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": avg-margin, f"{m}_upper": avg+margin})
        ci_list.append(m_row)

    pd.DataFrame(ci_list).to_csv(f"{PLOT_DIR}/confidence_intervals_detailed.csv", index=False)

    # WORST QUANTILE ANALYSIS (AVERAGE TREND)
    q_performance = detailed_df.groupby('quantile_q')[['CR_gpt', 'CR_base']].mean()
    worst_q_avg = q_performance['CR_gpt'].idxmin()
    worst_val_avg = q_performance['CR_gpt'].min()
    print(f"\nWorst Average Performance for GPT-4o occurs at q = {worst_q_avg:.4f} (CR: {worst_val_avg:.4f})")

    # Boxplot for Average CR
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('GPT-4o Decision Quality (Average CR)\n(Higher is Better; Line in box is Median)')
    plt.savefig(f"{PLOT_DIR}/1_comparison_boxplot_CR.png")
    plt.close()

    # Distribution of Worst Quantiles
    plt.figure(figsize=(8, 6))
    sns.violinplot(x='method', y='WorstCR_Quantile', data=summary_df, palette="Pastel1")
    plt.title('Distribution of Quantiles where Worst CR Occurs\n(Identifying failure points per item)')
    plt.savefig(f"{PLOT_DIR}/2_worst_quantile_distribution.png")
    plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      

>>> METHOD: Relevance Selection Method
Metric          |     Mean |   Median |           95% CI Range
-----------------------------------------------------------------
abs_error       |  31.1865 |  19.4064 | [26.4343, 35.9388]
squared_error   | 2730.3276 | 376.6143 | [1437.4006, 4023.2545]
AverageCR       |   0.6161 |   0.6810 | [ 0.5866,  0.6456]
WorstCR         |   0.3117 |   0.2652 | [ 0.2862,  0.3372]
Wasserstein     |  33.4716 |  22.6694 | [28.7566, 38.1866]
Kolmogorov      |   0.4910 |   0.4552 | [ 0.4660,  0.5160]

>>> METHOD: Baseline (Global)
Metric          |     Mean |   Median |           95% CI Range
-----------------------------------------------------------------
abs_error       |  34.1663 |  28.5608 | [29.6985, 38.6341]
squared_error   | 2720.9587 | 81

/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/4006983509.py:151: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/4006983509.py:158: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='method', y='WorstCR_Quantile', data=summary_df, palette="Pastel1")


#### GPT-5-mini

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import norm, wasserstein_distance, kstest, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'gpt_predictions_random_full_5_mini.csv'      
METRICS_OUT_FILE  = 'final_metrics_summary_descrip_5mini_newcal_CI_99.csv'        
DETAILED_OUT_FILE = 'final_metrics_detailed_descrip_5mini_newcal_CI_99.csv'       
PLOT_DIR          = 'plots_random_5mini_descri_newcal_CI_99'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

Q_VALUES = np.linspace(0.01, 0.99, 1000)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    loss = q * underage + (1 - q) * overage
    return np.mean(loss)

def get_discrete_quantile(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return 0
    idx = int(np.ceil(n * q)) - 1
    idx = max(0, min(idx, n - 1))
    return sorted_data[idx]

def get_normal_quantile(mu, sigma, q):
    if pd.isna(mu) or pd.isna(sigma): return 0
    if sigma <= 0: sigma = 0.001 
    val = norm.ppf(q, loc=mu, scale=sigma)
    return max(0, val)

# ==========================================
# MAIN LOGIC
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        gpt_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    global_pool = true_df[sales_cols].values.flatten()
    global_pool = np.sort(global_pool[~np.isnan(global_pool)])
    global_mean = np.mean(global_pool)

    summary_results, detailed_results = [], []
    unique_items = gpt_df['article_id'].unique()

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        true_row = true_df[true_df['article_id'] == art_id]
        if true_row.empty: continue
        
        true_demands = true_row[sales_cols].values.flatten()
        true_demands = true_demands[~np.isnan(true_demands)]
        true_mean, sorted_demands = np.mean(true_demands), np.sort(true_demands)

        gpt_row = gpt_df[gpt_df['article_id'] == art_id].iloc[0]
        pred_mean, pred_std = gpt_row.get('pred_mean'), gpt_row.get('pred_std')
        if pd.isna(pred_mean) or pd.isna(pred_std): continue

        synth_sigma = max(pred_std, 1e-3)
        gpt_synth_samples = np.maximum(0, np.random.normal(pred_mean, synth_sigma, size=5000))
        
        wass_gpt = wasserstein_distance(true_demands, gpt_synth_samples)
        wass_base = wasserstein_distance(true_demands, global_pool)
        ks_gpt, _ = kstest(true_demands, 'norm', args=(pred_mean, synth_sigma))
        ks_base, _ = ks_2samp(true_demands, global_pool)

        ratios_gpt, ratios_base = [], []
        for q in Q_VALUES:
            a_star = get_discrete_quantile(sorted_demands, q)
            a_gpt = get_normal_quantile(pred_mean, pred_std, q)
            a_base = get_discrete_quantile(global_pool, q)
            
            l_opt, l_gpt, l_base = [calculate_newsvendor_loss(a, true_demands, q) for a in [a_star, a_gpt, a_base]]
            
            r_gpt = min(l_opt / max(l_gpt, 1e-9), 1.0)
            r_base = min(l_opt / max(l_base, 1e-9), 1.0)
            
            ratios_gpt.append(r_gpt); ratios_base.append(r_base)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 4), 'CR_gpt': r_gpt, 'CR_base': r_base})

        worst_q_val_gpt = Q_VALUES[np.argmin(ratios_gpt)]
        worst_q_val_base = Q_VALUES[np.argmin(ratios_base)]

        # --- Added Squared Error Calculation ---
        sq_error_gpt = (pred_mean - true_mean) ** 2
        sq_error_base = (global_mean - true_mean) ** 2

        summary_results.append({
            'article_id': art_id, 'method': 'Relevance Selection Method', 'true_mean': true_mean, 'pred_mean': pred_mean,
            'abs_error': abs(pred_mean - true_mean), 
            'squared_error': sq_error_gpt, # Calculation included
            'AverageCR': np.mean(ratios_gpt), 
            'WorstCR': np.min(ratios_gpt), 'WorstCR_Quantile': worst_q_val_gpt,
            'Wasserstein': wass_gpt, 'Kolmogorov': ks_gpt
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': true_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - true_mean), 
            'squared_error': sq_error_base, # Calculation included
            'AverageCR': np.mean(ratios_base), 
            'WorstCR': np.min(ratios_base), 'WorstCR_Quantile': worst_q_val_base,
            'Wasserstein': wass_base, 'Kolmogorov': ks_base
        })

    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS WITH CONFIDENCE INTERVALS & MEDIAN
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*95)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      ")
    print("="*95)

    # Added 'squared_error' to the metrics list for aggregation
    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'Wasserstein', 'Kolmogorov']
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_data = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        method_row = {'Method': method}
        print(f"{'Metric':15} | {'Mean':>8} | {'Median':>8} | {'95% CI Range':>22}")
        print("-" * 75)
        
        for m in metrics:
            avg, med, std, n = stats_agg.loc[method, (m, ['mean', 'median', 'std', 'count'])]
            
            sem = std / np.sqrt(n)
            margin = 1.96 * sem
            lower, upper = avg - margin, avg + margin
            
            print(f"{m:15} | {avg:8.4f} | {med:8.4f} | [{lower:7.4f}, {upper:7.4f}]")
            method_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": lower, f"{m}_upper": upper})
        ci_data.append(method_row)

    pd.DataFrame(ci_data).to_csv(f"{PLOT_DIR}/confidence_intervals_summary.csv", index=False)

    # Plotting code remains the same...
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('Comparison of Decision Quality (Average CR)\n(Line inside box represents the Median)')
    plt.savefig(f"{PLOT_DIR}/1_comparison_boxplot_CR.png")
    plt.close()

    plt.figure(figsize=(10, 6))
    sns.violinplot(x='method', y='WorstCR_Quantile', data=summary_df, palette="Pastel1")
    plt.title('Distribution of Quantiles giving Worst CR\n(Concentration shows where model fails most often)')
    plt.savefig(f"{PLOT_DIR}/5_worst_quantile_distribution.png")
    plt.close()

    avg_metrics = detailed_df.groupby('quantile_q')[['CR_gpt', 'CR_base']].mean()
    plt.figure(figsize=(10, 5))
    plt.plot(avg_metrics.index, avg_metrics['CR_gpt'], label='Relevance Selection', color='blue')
    plt.plot(avg_metrics.index, avg_metrics['CR_base'], label='Baseline', color='red', linestyle='--')
    plt.xlabel("Critical Ratio (q)")
    plt.ylabel("Average Competitive Ratio")
    plt.legend(); plt.savefig(f"{PLOT_DIR}/4_average_cr_trend.png"); plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      

>>> METHOD: Relevance Selection Method
Metric          |     Mean |   Median |           95% CI Range
---------------------------------------------------------------------------
abs_error       |  31.1340 |  18.2214 | [26.1182, 36.1497]
squared_error   | 2920.8677 | 332.0205 | [1460.9537, 4380.7817]
AverageCR       |   0.6285 |   0.6889 | [ 0.5997,  0.6573]
WorstCR         |   0.3044 |   0.2490 | [ 0.2787,  0.3301]
Wasserstein     |  33.0576 |  20.6259 | [28.0873, 38.0280]
Kolmogorov      |   0.4869 |   0.4566 | [ 0.4618,  0.5120]

>>> METHOD: Baseline (Global)
Metric          |     Mean |   Median |           95% CI Range
---------------------------------------------------------------------------
abs_error       |  34.2107 |  28.5620 | [29.7288, 38.6926]
squared_erro

/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/1373714666.py:164: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/1373714666.py:170: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='method', y='WorstCR_Quantile', data=summary_df, palette="Pastel1")


#### Gemini

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import norm, wasserstein_distance, kstest, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'gemini_predictions_random_full_flash_new.csv'   
METRICS_OUT_FILE  = 'final_metrics_summary_gemini_newcal_CI_99.csv'        
DETAILED_OUT_FILE = 'final_metrics_detailed_gemini_newcal_CI_99.csv'       
PLOT_DIR          = 'plots_random_gemini_newcal_CI_99'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

# Restricted search range for WorstCR to avoid extreme numerical boundary noise
Q_VALUES = np.linspace(0.01, 0.99, 100)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    loss = q * underage + (1 - q) * overage
    return np.mean(loss)

def get_discrete_quantile(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return 0
    idx = int(np.ceil(n * q)) - 1
    idx = max(0, min(idx, n - 1))
    return sorted_data[idx]

def get_normal_quantile(mu, sigma, q):
    if pd.isna(mu) or pd.isna(sigma): return 0
    if sigma <= 0: sigma = 0.001 
    val = norm.ppf(q, loc=mu, scale=sigma)
    return max(0, val)

# ==========================================
# MAIN EVALUATION
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        gemini_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    global_pool = true_df[sales_cols].values.flatten()
    global_pool = np.sort(global_pool[~np.isnan(global_pool)])
    global_mean = np.mean(global_pool)

    summary_results, detailed_results = [], []
    unique_items = gemini_df['article_id'].unique()

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        true_row = true_df[true_df['article_id'] == art_id]
        if true_row.empty: continue
        
        true_demands = true_row[sales_cols].values.flatten()
        true_demands = true_demands[~np.isnan(true_demands)]
        true_mean, sorted_demands = np.mean(true_demands), np.sort(true_demands)

        g_row = gemini_df[gemini_df['article_id'] == art_id].iloc[0]
        pred_mean, pred_std = g_row.get('pred_mean'), g_row.get('pred_std')
        if pd.isna(pred_mean) or pd.isna(pred_std): continue

        synth_sigma = max(pred_std, 1e-3)
        gemini_synth_samples = np.maximum(0, np.random.normal(pred_mean, synth_sigma, size=5000))
        
        wass_g = wasserstein_distance(true_demands, gemini_synth_samples)
        wass_base = wasserstein_distance(true_demands, global_pool)
        ks_g, _ = kstest(true_demands, 'norm', args=(pred_mean, synth_sigma))
        ks_base, _ = ks_2samp(true_demands, global_pool)

        ratios_g, ratios_base = [], []
        qs_g, qs_base = [], []

        for q in Q_VALUES:
            a_star = get_discrete_quantile(sorted_demands, q)
            a_g = get_normal_quantile(pred_mean, pred_std, q)
            a_base = get_discrete_quantile(global_pool, q)
            
            l_opt, l_g, l_base = [calculate_newsvendor_loss(a, true_demands, q) for a in [a_star, a_g, a_base]]
            
            r_g = min(l_opt / max(l_g, 1e-9), 1.0)
            r_base = min(l_opt / max(l_base, 1e-9), 1.0)
            
            ratios_g.append(r_g); ratios_base.append(r_base)
            qs_g.append(q); qs_base.append(q)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 4), 'CR_g': r_g, 'CR_base': r_base})

        # Find the specific q that caused the worst CR for this article
        worst_idx_g = np.argmin(ratios_g)
        worst_idx_b = np.argmin(ratios_base)

        summary_results.append({
            'article_id': art_id, 'method': 'Descriptive Method', 'true_mean': true_mean, 'pred_mean': pred_mean,
            'abs_error': abs(pred_mean - true_mean), 'squared_error': (pred_mean - true_mean)**2, 
            'AverageCR': np.mean(ratios_g), 'WorstCR': np.min(ratios_g), 'WorstCase_Q': qs_g[worst_idx_g],
            'Wasserstein': wass_g, 'Kolmogorov': ks_g
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': true_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - true_mean), 'squared_error': (global_mean - true_mean)**2, 
            'AverageCR': np.mean(ratios_base), 'WorstCR': np.min(ratios_base), 'WorstCase_Q': qs_base[worst_idx_b],
            'Wasserstein': wass_base, 'Kolmogorov': ks_base
        })

    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS & VISUALIZATION
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*95)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      ")
    print("="*95)

    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'WorstCase_Q', 'Wasserstein', 'Kolmogorov']
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_data = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        method_row = {'Method': method}
        print(f"{'Metric':15} | {'Mean':>8} | {'Median':>8} | {'95% CI Range':>22}")
        print("-" * 75)

        for m in metrics:
            stats = stats_agg.loc[method, m]
            avg, med, std, n = stats['mean'], stats['median'], stats['std'], stats['count']
            sem = std / np.sqrt(n)
            margin = 1.96 * sem
            lower, upper = avg - margin, avg + margin
            
            print(f"{m:15} | {avg:8.4f} | {med:8.4f} | [{lower:7.4f}, {upper:7.4f}]")
            method_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": lower, f"{m}_upper": upper})
        ci_data.append(method_row)

    pd.DataFrame(ci_data).to_csv(f"{PLOT_DIR}/confidence_intervals_summary.csv", index=False)

    sns.set_theme(style="whitegrid")
    
    # Boxplot for CR
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('Comparison of Decision Quality (Average CR)\n(Horizontal line inside box = Median)')
    plt.savefig(f"{PLOT_DIR}/1_comparison_boxplot_CR.png")
    plt.close()

    # Violin Plot for Worst-Case Quantiles (Failure Profile)
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, inner="quartile", palette="Pastel1")
    plt.title('Distribution of Quantiles where Model Performance Fails (WorstCase_Q)')
    plt.ylabel("Quantile q")
    plt.savefig(f"{PLOT_DIR}/2_worst_quantile_profile.png")
    plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      

>>> METHOD: Descriptive Method
Metric          |     Mean |   Median |           95% CI Range
---------------------------------------------------------------------------
abs_error       |  33.7137 |  21.6171 | [29.0076, 38.4197]
squared_error   | 2860.3554 | 467.4048 | [1802.8340, 3917.8768]
AverageCR       |   0.6033 |   0.6557 | [ 0.5727,  0.6338]
WorstCR         |   0.3125 |   0.2580 | [ 0.2860,  0.3391]
WorstCase_Q     |   0.7410 |   0.9900 | [ 0.7005,  0.7815]
Wasserstein     |  36.1278 |  23.6167 | [31.4645, 40.7912]
Kolmogorov      |   0.4964 |   0.4731 | [ 0.4710,  0.5217]

>>> METHOD: Baseline (Global)
Metric          |     Mean |   Median |           95% CI Range
---------------------------------------------------------------------------
abs_error       |  3

/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/2965524008.py:161: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/2965524008.py:168: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, inner="quartile", palette="Pastel1")


#### Mistral

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import norm, wasserstein_distance, kstest, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'mistral_predictions_descriptive.csv'   
METRICS_OUT_FILE  = 'final_metrics_summary_descrip_mistral_newcal_CI_99.csv'        
DETAILED_OUT_FILE = 'final_metrics_detailed_descrip_mistral_newcal_CI_99.csv'       
PLOT_DIR          = 'plots_random_mistral_descri_newcal_CI_99'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

Q_VALUES = np.linspace(0.01, 0.99, 1000)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    loss = q * underage + (1 - q) * overage
    return np.mean(loss)

def get_discrete_quantile(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return 0
    idx = int(np.ceil(n * q)) - 1
    idx = max(0, min(idx, n - 1))
    return sorted_data[idx]

def get_normal_quantile(mu, sigma, q):
    if pd.isna(mu) or pd.isna(sigma): return 0
    if sigma <= 0: sigma = 0.001 
    val = norm.ppf(q, loc=mu, scale=sigma)
    return max(0, val)

# ==========================================
# MAIN EVALUATION
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        mistral_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    global_pool = true_df[sales_cols].values.flatten()
    global_pool = np.sort(global_pool[~np.isnan(global_pool)])
    global_mean = np.mean(global_pool)

    summary_results, detailed_results = [], []
    unique_items = mistral_df['article_id'].unique()

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        true_row = true_df[true_df['article_id'] == art_id]
        if true_row.empty: continue
        
        true_demands = true_row[sales_cols].values.flatten()
        true_demands = true_demands[~np.isnan(true_demands)]
        true_mean, sorted_demands = np.mean(true_demands), np.sort(true_demands)

        m_row = mistral_df[mistral_df['article_id'] == art_id].iloc[0]
        pred_mean, pred_std = m_row.get('pred_mean'), m_row.get('pred_std')
        if pd.isna(pred_mean) or pd.isna(pred_std): continue

        synth_sigma = max(pred_std, 1e-3)
        mistral_synth_samples = np.maximum(0, np.random.normal(pred_mean, synth_sigma, size=5000))
        
        wass_m = wasserstein_distance(true_demands, mistral_synth_samples)
        wass_base = wasserstein_distance(true_demands, global_pool)
        ks_m, _ = kstest(true_demands, 'norm', args=(pred_mean, synth_sigma))
        ks_base, _ = ks_2samp(true_demands, global_pool)

        ratios_m, ratios_base = [], []
        for q in Q_VALUES:
            a_star = get_discrete_quantile(sorted_demands, q)
            a_m = get_normal_quantile(pred_mean, pred_std, q)
            a_base = get_discrete_quantile(global_pool, q)
            
            l_opt, l_m, l_base = [calculate_newsvendor_loss(a, true_demands, q) for a in [a_star, a_m, a_base]]
            
            r_m = min(l_opt / max(l_m, 1e-9), 1.0)
            r_base = min(l_opt / max(l_base, 1e-9), 1.0)
            
            ratios_m.append(r_m); ratios_base.append(r_base)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 4), 'CR_m': r_m, 'CR_base': r_base})

        # Calculate squared error
        sq_err_m = (pred_mean - true_mean) ** 2
        sq_err_base = (global_mean - true_mean) ** 2

        summary_results.append({
            'article_id': art_id, 'method': 'Descriptive Method', 'true_mean': true_mean, 'pred_mean': pred_mean,
            'abs_error': abs(pred_mean - true_mean), 
            'squared_error': sq_err_m, # Added calculation
            'AverageCR': np.mean(ratios_m), 
            'WorstCR': np.min(ratios_m), 'WorstCR_Quantile': Q_VALUES[np.argmin(ratios_m)],
            'Wasserstein': wass_m, 'Kolmogorov': ks_m
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': true_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - true_mean), 
            'squared_error': sq_err_base, # Added calculation
            'AverageCR': np.mean(ratios_base), 
            'WorstCR': np.min(ratios_base), 'WorstCR_Quantile': Q_VALUES[np.argmin(ratios_base)],
            'Wasserstein': wass_base, 'Kolmogorov': ks_base
        })

    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS WITH CONFIDENCE INTERVALS & MEDIAN
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*95)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      ")
    print("="*95)

    # Added squared_error to metrics list
    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'Wasserstein', 'Kolmogorov']
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_data = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        method_row = {'Method': method}
        print(f"{'Metric':15} | {'Mean':>8} | {'Median':>8} | {'95% CI Range':>22}")
        print("-" * 75)

        for m in metrics:
            avg, med, std, n = stats_agg.loc[method, (m, ['mean', 'median', 'std', 'count'])]
            
            sem = std / np.sqrt(n)
            margin = 1.96 * sem
            lower, upper = avg - margin, avg + margin
            
            print(f"{m:15} | {avg:8.4f} | {med:8.4f} | [{lower:7.4f}, {upper:7.4f}]")
            method_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": lower, f"{m}_upper": upper})
        ci_data.append(method_row)

    pd.DataFrame(ci_data).to_csv(f"{PLOT_DIR}/confidence_intervals_detailed.csv", index=False)

    # Plotting
    sns.set_theme(style="whitegrid")
    
    # Boxplot for Average CR
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('Comparison of Decision Quality (Average CR)\n(Line inside box represents the Median)')
    plt.savefig(f"{PLOT_DIR}/1_comparison_boxplot_CR.png")
    plt.close()

    # Violin plot for WorstCR_Quantile
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='method', y='WorstCR_Quantile', data=summary_df, palette="Pastel1")
    plt.title('Distribution of Quantiles giving Worst CR')
    plt.savefig(f"{PLOT_DIR}/5_worst_quantile_distribution.png")
    plt.close()

    # Trend Plot
    avg_metrics = detailed_df.groupby('quantile_q')[['CR_m', 'CR_base']].mean()
    plt.figure(figsize=(10, 5))
    plt.plot(avg_metrics.index, avg_metrics['CR_m'], label='Descriptive Method', color='blue')
    plt.plot(avg_metrics.index, avg_metrics['CR_base'], label='Baseline', color='red', linestyle='--')
    plt.xlabel("Critical Ratio (q)")
    plt.ylabel("Average Competitive Ratio")
    plt.legend(); plt.savefig(f"{PLOT_DIR}/4_average_cr_trend.png"); plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      

>>> METHOD: Descriptive Method
Metric          |     Mean |   Median |           95% CI Range
---------------------------------------------------------------------------
abs_error       |  29.5748 |  18.4420 | [25.1642, 33.9854]
squared_error   | 2388.7485 | 340.1113 | [1184.7838, 3592.7132]
AverageCR       |   0.6347 |   0.7064 | [ 0.6054,  0.6640]
WorstCR         |   0.3328 |   0.2972 | [ 0.3057,  0.3598]
Wasserstein     |  31.4278 |  21.7342 | [27.1085, 35.7471]
Kolmogorov      |   0.4739 |   0.4504 | [ 0.4491,  0.4988]

>>> METHOD: Baseline (Global)
Metric          |     Mean |   Median |           95% CI Range
---------------------------------------------------------------------------
abs_error       |  34.1663 |  28.5608 | [29.6985, 38.6341]
squared_error   | 27

/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/3457760844.py:163: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/3457760844.py:170: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='method', y='WorstCR_Quantile', data=summary_df, palette="Pastel1")


### Relevant

#### GPT-4o

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
from scipy.stats import wasserstein_distance, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'gpt_predictions_random100_unconstrained.csv'     
METRICS_OUT_FILE  = 'final_metrics_summary_empirical_99.csv'
DETAILED_OUT_FILE = 'final_metrics_detailed_empirical_99.csv' 
PLOT_DIR          = 'plots_empirical_4o_99'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

# Quantiles for the sampled Average CR and plotting
Q_VALUES_PLOT = np.linspace(0.01, 0.99, 99)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    decision_a = np.array(decision_a)
    demands_xi = np.array(demands_xi)
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    loss = q * underage + (1 - q) * overage
    return np.mean(loss)

def get_discrete_quantile_candidates(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return [0]
    val = n * q
    if np.isclose(val, np.round(val), atol=1e-9):
        idx = int(np.round(val))
        lower_idx = max(0, idx - 1)
        upper_idx = min(n - 1, idx)
        return list(set([sorted_data[lower_idx], sorted_data[upper_idx]]))
    else:
        idx = int(np.ceil(val)) - 1
        return [sorted_data[max(0, min(idx, n - 1))]]

def calculate_worst_cr_theoretical(true_demands, pred_samples, q_min=0.01, q_max=0.99):
    m, m_hat = len(true_demands), len(pred_samples)
    if m == 0 or m_hat == 0: return 0.0, 0.5
    
    # Generate critical points from both distributions
    q_set_true = [i/m for i in range(1, m)]
    q_set_pred = [j/m_hat for j in range(1, m_hat)]
    
    # RESTRICT Q to [0.01, 0.99] as requested
    critical_qs = [q for q in set(q_set_true + q_set_pred) if q_min <= q <= q_max]
    
    if not critical_qs: 
        critical_qs = [q_min, q_max]
    else:
        critical_qs = sorted(critical_qs)

    min_cr = 1.0 
    worst_q = 0.5
    true_sorted = np.sort(true_demands)
    pred_sorted = np.sort(pred_samples)

    for q in critical_qs:
        opts = get_discrete_quantile_candidates(true_sorted, q)
        loss_opt = calculate_newsvendor_loss(opts[0], true_demands, q)
        
        pred_candidates = get_discrete_quantile_candidates(pred_sorted, q)
        # Take the worst possible performance among discrete candidates for a conservative estimate
        max_pred_loss = max([calculate_newsvendor_loss(ha, true_demands, q) for ha in pred_candidates])
        
        cr = 1.0 if max_pred_loss < 1e-9 else loss_opt / max_pred_loss
        
        if cr < min_cr: 
            min_cr = cr
            worst_q = q
            
    return min_cr, worst_q

# ==========================================
# MAIN EVALUATION
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        gpt_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    global_pool = true_df[sales_cols].values.flatten()
    global_pool = np.sort(global_pool[~np.isnan(global_pool)])
    global_mean = np.mean(global_pool)

    summary_results, detailed_results = [], []
    unique_items = gpt_df['article_id'].unique()

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        true_row = true_df[true_df['article_id'] == art_id]
        if true_row.empty: continue
        
        true_demands = true_row[sales_cols].values.flatten()
        true_demands = true_demands[~np.isnan(true_demands)]
        true_mean, sorted_demands = np.mean(true_demands), np.sort(true_demands)

        gpt_row = gpt_df[gpt_df['article_id'] == art_id].iloc[0]
        try:
            raw_emp_data = gpt_row.get('empirical_sales_data', '[]')
            pred_samples = np.array([x for x in ast.literal_eval(str(raw_emp_data)) if pd.notna(x)])
            pred_samples.sort()
        except: pred_samples = np.array([])
        
        if len(pred_samples) == 0: continue
        pred_mean = np.mean(pred_samples)

        # Theoretical Worst CR Restricted to [0.01, 0.99]
        worst_cr_gpt, worst_q_gpt = calculate_worst_cr_theoretical(true_demands, pred_samples)
        worst_cr_base, worst_q_base = calculate_worst_cr_theoretical(true_demands, global_pool)

        # Standard Metrics
        wass_gpt = wasserstein_distance(true_demands, pred_samples)
        ks_gpt = ks_2samp(true_demands, pred_samples).statistic
        wass_base = wasserstein_distance(true_demands, global_pool)
        ks_base = ks_2samp(true_demands, global_pool).statistic

        # Sampled CR for Average Calculation
        ratios_gpt, ratios_base = [], []
        for q in Q_VALUES_PLOT:
            a_star = get_discrete_quantile_candidates(sorted_demands, q)[0]
            a_gpt  = get_discrete_quantile_candidates(pred_samples, q)[0]
            a_base = get_discrete_quantile_candidates(global_pool, q)[0]
            
            l_opt, l_gpt, l_base = [calculate_newsvendor_loss(a, true_demands, q) for a in [a_star, a_gpt, a_base]]
            r_gpt = (l_opt / l_gpt) if l_gpt > 1e-9 else 1.0
            r_base = (l_opt / l_base) if l_base > 1e-9 else 1.0
            
            ratios_gpt.append(r_gpt); ratios_base.append(r_base)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 2), 'CR_gpt': r_gpt, 'CR_base': r_base})

        summary_results.append({
            'article_id': art_id, 'method': 'Relevance Selection Method', 'true_mean': true_mean, 'pred_mean': pred_mean,
            'abs_error': abs(pred_mean - true_mean), 'squared_error': (pred_mean - true_mean)**2,
            'AverageCR': np.mean(ratios_gpt), 'WorstCR': worst_cr_gpt, 'WorstCase_Q': worst_q_gpt,
            'Wasserstein': wass_gpt, 'Kolmogorov': ks_gpt
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': true_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - true_mean), 'squared_error': (global_mean - true_mean)**2,
            'AverageCR': np.mean(ratios_base), 'WorstCR': worst_cr_base, 'WorstCase_Q': worst_q_base,
            'Wasserstein': wass_base, 'Kolmogorov': ks_base
        })

    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS AND PLOTTING
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*95)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      ")
    print("="*95)

    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'Wasserstein', 'Kolmogorov']
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_data = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        method_row = {'Method': method}
        print(f"{'Metric':15} | {'Mean':>8} | {'Median':>8} | {'95% CI Range':>22}")
        print("-" * 75)
        for m in metrics:
            avg, med, std, n = stats_agg.loc[method, (m, 'mean')], stats_agg.loc[method, (m, 'median')], stats_agg.loc[method, (m, 'std')], stats_agg.loc[method, (m, 'count')]
            sem = std / np.sqrt(n)
            margin = 1.96 * sem
            lower, upper = avg - margin, avg + margin
            
            print(f"{m:15} | {avg:8.4f} | {med:8.4f} | [{lower:7.4f}, {upper:7.4f}]")
            method_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": lower, f"{m}_upper": upper})
        ci_data.append(method_row)

    pd.DataFrame(ci_data).to_csv(f"{PLOT_DIR}/confidence_intervals_summary.csv", index=False)

    sns.set_theme(style="whitegrid")
    
    # Boxplot for CR
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('Comparison of Decision Quality (Average CR)')
    plt.savefig(f"{PLOT_DIR}/1_boxplot_AverageCR.png")
    plt.close()

    # Violin plot for Worst-Case Quantile (Failure Analysis)
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, palette="Pastel1", inner="quartile")
    plt.title('Distribution of Quantiles giving Worst CR\n(Restricted to [0.01, 0.99])')
    plt.savefig(f"{PLOT_DIR}/2_worst_quantile_dist.png")
    plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      

>>> METHOD: Relevance Selection Method
Metric          |     Mean |   Median |           95% CI Range
---------------------------------------------------------------------------
abs_error       |  34.0555 |  25.2825 | [29.7916, 38.3194]
squared_error   | 2574.8198 | 639.2062 | [1548.3700, 3601.2697]
AverageCR       |   0.6452 |   0.6920 | [ 0.6188,  0.6716]
WorstCR         |   0.2755 |   0.2083 | [ 0.2508,  0.3002]
Wasserstein     |  38.1073 |  29.2540 | [33.9008, 42.3138]
Kolmogorov      |   0.4228 |   0.4016 | [ 0.4019,  0.4438]

>>> METHOD: Baseline (Global)
Metric          |     Mean |   Median |           95% CI Range
---------------------------------------------------------------------------
abs_error       |  34.1663 |  28.5608 | [29.6985, 38.6341]
squared_erro

/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/1731490332.py:200: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/1731490332.py:207: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, palette="Pastel1", inner="quartile")


#### GPT-5-mini

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
from scipy.stats import wasserstein_distance, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'gpt_predictions_random100_unconstrained_5mini.csv'     
METRICS_OUT_FILE  = 'final_metrics_summary_empirical_5mini_newcal_CI_99.csv'
DETAILED_OUT_FILE = 'final_metrics_detailed_empirical_5mini_newcal_CI_99.csv'
PLOT_DIR          = 'plots_random_5mini_empirical_newcal_CI_99'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

Q_VALUES_PLOT = np.linspace(0.01, 0.99, 99)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    decision_a = np.array(decision_a)
    demands_xi = np.array(demands_xi)
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    loss = q * underage + (1 - q) * overage
    return np.mean(loss)

def get_discrete_quantile_candidates(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return [0]
    val = n * q
    if np.isclose(val, np.round(val), atol=1e-9):
        idx = int(np.round(val))
        l_idx, u_idx = max(0, idx - 1), min(n - 1, idx)
        return list(set([sorted_data[l_idx], sorted_data[u_idx]]))
    else:
        return [sorted_data[max(0, min(int(np.ceil(val)) - 1, n - 1))]]

def calculate_worst_cr_theoretical(true_demands, pred_samples, q_min=0.01, q_max=0.99):
    """
    Finds the absolute minimum Competitive Ratio within a restricted range.
    Identifies both the value and the specific q (Worst-Case Quantile).
    """
    m, m_hat = len(true_demands), len(pred_samples)
    if m == 0 or m_hat == 0: return 0.0, 0.5
    
    # Generate critical points where the empirical CDF jumps
    all_qs = [i/m for i in range(1, m)] + [j/m_hat for j in range(1, m_hat)]
    
    # Filter for the operational range [0.01, 0.99]
    critical_qs = sorted(list(set([q for q in all_qs if q_min <= q <= q_max])))
    if not critical_qs: critical_qs = [q_min, q_max]

    min_cr = 1.0 
    worst_q = 0.5
    true_sorted, pred_sorted = np.sort(true_demands), np.sort(pred_samples)

    for q in critical_qs:
        loss_opt = calculate_newsvendor_loss(get_discrete_quantile_candidates(true_sorted, q)[0], true_demands, q)
        pred_cands = get_discrete_quantile_candidates(pred_sorted, q)
        
        # Max pred loss among candidates ensures a conservative (true worst-case) estimate
        max_pred_loss = max([calculate_newsvendor_loss(ha, true_demands, q) for ha in pred_cands])
        
        cr = 1.0 if max_pred_loss < 1e-9 else loss_opt / max_pred_loss
        if cr < min_cr: 
            min_cr = cr
            worst_q = q
    return min_cr, worst_q

# ==========================================
# MAIN LOGIC
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        gpt_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    global_pool = np.sort(true_df[sales_cols].values.flatten()[~np.isnan(true_df[sales_cols].values.flatten())])
    global_mean = np.mean(global_pool)

    summary_results, detailed_results = [], []
    unique_items = gpt_df['article_id'].unique()

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        true_row = true_df[true_df['article_id'] == art_id]
        if true_row.empty: continue
        true_demands = np.sort(true_row[sales_cols].values.flatten()[~np.isnan(true_row[sales_cols].values.flatten())])
        true_mean = np.mean(true_demands)

        gpt_row = gpt_df[gpt_df['article_id'] == art_id].iloc[0]
        try:
            pred_samples = np.sort([x for x in ast.literal_eval(str(gpt_row.get('empirical_sales_data', '[]'))) if pd.notna(x)])
        except: pred_samples = np.array([])
        
        if len(pred_samples) == 0: continue
        pred_mean = np.mean(pred_samples)

        # Theoretical metrics restricted to operational range
        worst_cr_gpt, worst_q_gpt = calculate_worst_cr_theoretical(true_demands, pred_samples)
        worst_cr_base, worst_q_base = calculate_worst_cr_theoretical(true_demands, global_pool)

        ratios_gpt, ratios_base = [], []
        for q in Q_VALUES_PLOT:
            a_star = get_discrete_quantile_candidates(true_demands, q)[0]
            a_gpt  = get_discrete_quantile_candidates(pred_samples, q)[0]
            a_base = get_discrete_quantile_candidates(global_pool, q)[0]
            
            l_opt, l_gpt, l_base = [calculate_newsvendor_loss(a, true_demands, q) for a in [a_star, a_gpt, a_base]]
            r_gpt = (l_opt / l_gpt) if l_gpt > 1e-9 else 1.0
            r_base = (l_opt / l_base) if l_base > 1e-9 else 1.0
            
            ratios_gpt.append(r_gpt); ratios_base.append(r_base)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 2), 'CR_gpt': r_gpt, 'CR_base': r_base})

        summary_results.append({
            'article_id': art_id, 'method': 'Relevance Selection Method', 'true_mean': true_mean, 'pred_mean': pred_mean, 
            'abs_error': abs(pred_mean - true_mean), 'squared_error': (pred_mean - true_mean)**2,
            'AverageCR': np.mean(ratios_gpt), 'WorstCR': worst_cr_gpt, 'WorstCase_Q': worst_q_gpt,
            'Wasserstein': wasserstein_distance(true_demands, pred_samples), 'Kolmogorov': ks_2samp(true_demands, pred_samples).statistic
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': true_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - true_mean), 'squared_error': (global_mean - true_mean)**2,
            'AverageCR': np.mean(ratios_base), 'WorstCR': worst_cr_base, 'WorstCase_Q': worst_q_base,
            'Wasserstein': wasserstein_distance(true_demands, global_pool), 'Kolmogorov': ks_2samp(true_demands, global_pool).statistic
        })

    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS WITH MEAN, MEDIAN, AND CI
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*95)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CONFIDENCE INTERVALS)      ")
    print("="*95)

    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'Wasserstein', 'Kolmogorov']
    # Aggregate Mean, Median, Std, and Count
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_data = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        method_row = {'Method': method}
        print(f"{'Metric':15} | {'Mean':>8} | {'Median':>8} | {'95% CI Range':>20}")
        print("-" * 75)
        for m in metrics:
            avg, med, std, n = stats_agg.loc[method, (m, 'mean')], stats_agg.loc[method, (m, 'median')], stats_agg.loc[method, (m, 'std')], stats_agg.loc[method, (m, 'count')]
            sem = std / np.sqrt(n)
            margin = 1.96 * sem
            lower, upper = avg - margin, avg + margin
            print(f"{m:15} | {avg:8.4f} | {med:8.4f} | [{lower:6.4f}, {upper:6.4f}]")
            method_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": lower, f"{m}_upper": upper})
        ci_data.append(method_row)

    pd.DataFrame(ci_data).to_csv(f"{PLOT_DIR}/confidence_intervals.csv", index=False)

    sns.set_theme(style="whitegrid")
    
    # Boxplot for CR (Distribution and Median)
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('Comparison of Decision Quality (Average CR)\n(Line inside box represents the Median)')
    plt.savefig(f"{PLOT_DIR}/1_comparison_boxplot_CR.png")
    plt.close()

    # Failure Analysis: Distribution of Worst-Case Quantiles
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, inner="quartile", palette="Pastel1")
    plt.title('Distribution of Worst-Case Quantiles (q) in [0.01, 0.99]\n(Identifies which service levels are most difficult to predict)')
    plt.savefig(f"{PLOT_DIR}/2_worst_case_q_distribution.png")
    plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CONFIDENCE INTERVALS)      

>>> METHOD: Relevance Selection Method
Metric          |     Mean |   Median |         95% CI Range
---------------------------------------------------------------------------
abs_error       |  33.3463 |  27.1152 | [29.0730, 37.6195]
squared_error   | 2533.2530 | 735.2378 | [1347.8935, 3718.6126]
AverageCR       |   0.6503 |   0.6902 | [0.6253, 0.6752]
WorstCR         |   0.2369 |   0.1809 | [0.2161, 0.2577]
Wasserstein     |  38.3332 |  32.6240 | [34.1646, 42.5019]
Kolmogorov      |   0.4103 |   0.3722 | [0.3903, 0.4303]

>>> METHOD: Baseline (Global)
Metric          |     Mean |   Median |         95% CI Range
---------------------------------------------------------------------------
abs_error       |  34.1663 |  28.5608 | [29.6985, 38.6341]
squa

/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/2127439476.py:180: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/2127439476.py:187: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, inner="quartile", palette="Pastel1")


#### Gemini

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
from scipy.stats import wasserstein_distance, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'gemini_2.0_predictions_random100_unconstrained_new.csv'     
METRICS_OUT_FILE  = 'final_metrics_summary_empirical_gemini_newcal_CI_9999.csv'
DETAILED_OUT_FILE = 'final_metrics_detailed_empirical_gemini_newcal_CI_9999.csv'
PLOT_DIR          = 'plots_random_gemini_empirical_newcal_CI_9999'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

Q_VALUES_PLOT = np.linspace(0.01, 0.9999, 99)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    decision_a = np.array(decision_a)
    demands_xi = np.array(demands_xi)
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    return np.mean(q * underage + (1 - q) * overage)

def get_discrete_quantile_candidates(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return [0]
    val = n * q
    if np.isclose(val, np.round(val), atol=1e-9):
        idx = int(np.round(val))
        l_idx, u_idx = max(0, idx - 1), min(n - 1, idx)
        return list(set([sorted_data[l_idx], sorted_data[u_idx]]))
    else:
        return [sorted_data[max(0, min(int(np.ceil(val)) - 1, n - 1))]]

def calculate_worst_cr_theoretical(true_demands, pred_samples, q_min=0.01, q_max=0.9999):
    m, m_hat = len(true_demands), len(pred_samples)
    if m == 0 or m_hat == 0: return 0.0, 0.5
    
    # Generate critical points and restrict to operational range
    all_qs = [i/m for i in range(1, m)] + [j/m_hat for j in range(1, m_hat)]
    critical_qs = sorted(list(set([q for q in all_qs if q_min <= q <= q_max])))
    
    if not critical_qs: critical_qs = [q_min, q_max]

    min_cr = 1.0 
    worst_q = 0.5
    ts, ps = np.sort(true_demands), np.sort(pred_samples)

    for q in critical_qs:
        l_opt = calculate_newsvendor_loss(get_discrete_quantile_candidates(ts, q)[0], true_demands, q)
        # Check all discrete candidates for the prediction to find true worst-case
        l_pred_max = max([calculate_newsvendor_loss(ha, true_demands, q) for ha in get_discrete_quantile_candidates(ps, q)])
        
        cr = 1.0 if l_pred_max < 1e-9 else l_opt / l_pred_max
        if cr < min_cr: 
            min_cr = cr
            worst_q = q
    return min_cr, worst_q

# ==========================================
# MAIN LOGIC
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        gemini_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    global_pool = np.sort(true_df[sales_cols].values.flatten()[~np.isnan(true_df[sales_cols].values.flatten())])
    global_mean = np.mean(global_pool)

    summary_results, detailed_results = [], []
    unique_items = gemini_df['article_id'].unique()

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        t_row = true_df[true_df['article_id'] == art_id]
        if t_row.empty: continue
        t_demands = np.sort(t_row[sales_cols].values.flatten()[~np.isnan(t_row[sales_cols].values.flatten())])
        t_mean = np.mean(t_demands)

        g_row = gemini_df[gemini_df['article_id'] == art_id].iloc[0]
        try:
            p_samples = np.sort([x for x in ast.literal_eval(str(g_row.get('empirical_sales_data', '[]'))) if pd.notna(x)])
        except: p_samples = np.array([])
        
        if len(p_samples) == 0: continue
        p_mean = np.mean(p_samples)

        # Restricted Theoretical Worst CR
        worst_cr_gem, worst_q_gem = calculate_worst_cr_theoretical(t_demands, p_samples)
        worst_cr_base, worst_q_base = calculate_worst_cr_theoretical(t_demands, global_pool)

        r_gem, r_base = [], []
        for q in Q_VALUES_PLOT:
            a_star, a_gem, a_base = [get_discrete_quantile_candidates(d, q)[0] for d in [t_demands, p_samples, global_pool]]
            lo, lg, lb = [calculate_newsvendor_loss(a, t_demands, q) for a in [a_star, a_gem, a_base]]
            rg, rb = [(lo/x if x > 1e-9 else 1.0) for x in [lg, lb]]
            r_gem.append(rg); r_base.append(rb)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 2), 'CR_gem': rg, 'CR_base': rb})

        summary_results.append({
            'article_id': art_id, 'method': 'Relevance Selection Method', 'true_mean': t_mean, 'pred_mean': p_mean,
            'abs_error': abs(p_mean - t_mean), 'squared_error': (p_mean - t_mean)**2,
            'AverageCR': np.mean(r_gem), 'WorstCR': worst_cr_gem, 'WorstCase_Q': worst_q_gem,
            'Wasserstein': wasserstein_distance(t_demands, p_samples), 'Kolmogorov': ks_2samp(t_demands, p_samples).statistic
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': t_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - t_mean), 'squared_error': (global_mean - t_mean)**2,
            'AverageCR': np.mean(r_base), 'WorstCR': worst_cr_base, 'WorstCase_Q': worst_q_base,
            'Wasserstein': wasserstein_distance(t_demands, global_pool), 'Kolmogorov': ks_2samp(t_demands, global_pool).statistic
        })

    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS & 95% CI
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*95)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      ")
    print("="*95)

    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'Wasserstein', 'Kolmogorov']
    # Aggregating median along with mean and std
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_data = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        print(f"{'Metric':15} | {'Mean':>8} | {'Median':>8} | {'95% CI Range':>20}")
        print("-" * 75)
        m_row = {'Method': method}
        for m in metrics:
            avg, med, std, n = stats_agg.loc[method, (m, 'mean')], stats_agg.loc[method, (m, 'median')], stats_agg.loc[method, (m, 'std')], stats_agg.loc[method, (m, 'count')]
            sem = std / np.sqrt(n)
            margin = 1.96 * sem
            lower, upper = avg - margin, avg + margin
            print(f"{m:15} | {avg:8.4f} | {med:8.4f} | [{lower:6.4f}, {upper:6.4f}]")
            m_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": lower, f"{m}_upper": upper})
        ci_data.append(m_row)

    pd.DataFrame(ci_data).to_csv(f"{PLOT_DIR}/confidence_intervals.csv", index=False)

    sns.set_theme(style="whitegrid")
    
    # Boxplot for Average CR (Visible median)
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('Gemini 2.0: Decision Quality (Average CR)\n(Line inside box represents the Median)')
    plt.savefig(f"{PLOT_DIR}/1_comparison_boxplot_CR.png")
    plt.close()

    # Violin plot for Worst-Case Quantile (Failure Analysis)
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, inner="quartile", palette="Pastel1")
    plt.title('Distribution of Worst-Case Quantiles (q) in [0.01, 0.99]\n(Identifies where the model fails most)')
    plt.savefig(f"{PLOT_DIR}/2_worst_case_q_distribution.png")
    plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      

>>> METHOD: Relevance Selection Method
Metric          |     Mean |   Median |         95% CI Range
---------------------------------------------------------------------------
abs_error       |  33.2891 |  25.9327 | [28.9474, 37.6307]
squared_error   | 2575.3162 | 672.5282 | [1363.6437, 3786.9887]
AverageCR       |   0.6527 |   0.7037 | [0.6279, 0.6775]
WorstCR         |   0.1925 |   0.1256 | [0.1726, 0.2125]
Wasserstein     |  37.9291 |  33.6999 | [33.6725, 42.1857]
Kolmogorov      |   0.4086 |   0.3788 | [0.3886, 0.4287]

>>> METHOD: Baseline (Global)
Metric          |     Mean |   Median |         95% CI Range
---------------------------------------------------------------------------
abs_error       |  34.1663 |  28.5608 | [29.6985, 38.6341]
squared_error   | 2720

/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/1621936313.py:167: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/1621936313.py:174: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, inner="quartile", palette="Pastel1")


#### Mistral

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
from scipy.stats import wasserstein_distance, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'mistral_large_predictions_random100_unconstrained.csv'    
METRICS_OUT_FILE  = 'final_metrics_summary_empirical_mistral_newcal_CI_99.csv'
DETAILED_OUT_FILE = 'final_metrics_detailed_empirical_mistral_newcal_CI_99.csv'
PLOT_DIR          = 'plots_random_mistral_empirical_newcal_CI_99'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

Q_VALUES_PLOT = np.linspace(0.01, 0.99, 99)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    decision_a = np.array(decision_a)
    demands_xi = np.array(demands_xi)
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    return np.mean(q * underage + (1 - q) * overage)

def get_discrete_quantile_candidates(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return [0]
    val = n * q
    if np.isclose(val, np.round(val), atol=1e-9):
        idx = int(np.round(val))
        l_idx, u_idx = max(0, idx - 1), min(n - 1, idx)
        return list(set([sorted_data[l_idx], sorted_data[u_idx]]))
    else:
        return [sorted_data[max(0, min(int(np.ceil(val)) - 1, n - 1))]]

def calculate_worst_cr_theoretical(true_demands, pred_samples, q_min=0.01, q_max=0.99):
    """
    Finds the absolute minimum Competitive Ratio within a restricted range.
    Returns (min_cr, worst_q).
    """
    m, m_hat = len(true_demands), len(pred_samples)
    if m == 0 or m_hat == 0: return 0.0, 0.5
    
    # Generate critical points and restrict to [0.01, 0.99]
    all_qs = [i/m for i in range(1, m)] + [j/m_hat for j in range(1, m_hat)]
    critical_qs = sorted(list(set([q for q in all_qs if q_min <= q <= q_max])))
    
    if not critical_qs: critical_qs = [q_min, q_max]

    min_cr = 1.0 
    worst_q = 0.5
    true_s, pred_s = np.sort(true_demands), np.sort(pred_samples)

    for q in critical_qs:
        l_opt = calculate_newsvendor_loss(get_discrete_quantile_candidates(true_s, q)[0], true_demands, q)
        # Take max loss among discrete candidates for a conservative worst-case
        l_pred_max = max([calculate_newsvendor_loss(ha, true_demands, q) for ha in get_discrete_quantile_candidates(pred_s, q)])
        
        cr = 1.0 if l_pred_max < 1e-9 else l_opt / l_pred_max
        if cr < min_cr: 
            min_cr = cr
            worst_q = q
    return min_cr, worst_q

# ==========================================
# MAIN EVALUATION
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        mistral_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    global_pool = np.sort(true_df[sales_cols].values.flatten()[~np.isnan(true_df[sales_cols].values.flatten())])
    global_mean = np.mean(global_pool)

    summary_results, detailed_results = [], []
    unique_items = mistral_df['article_id'].unique()

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        t_row = true_df[true_df['article_id'] == art_id]
        if t_row.empty: continue
        t_demands = np.sort(t_row[sales_cols].values.flatten()[~np.isnan(t_row[sales_cols].values.flatten())])
        t_mean = np.mean(t_demands)

        m_row = mistral_df[mistral_df['article_id'] == art_id].iloc[0]
        try:
            p_samples = np.sort([x for x in ast.literal_eval(str(m_row.get('empirical_sales_data', '[]'))) if pd.notna(x)])
        except: p_samples = np.array([])
        
        if len(p_samples) == 0: continue
        p_mean = np.mean(p_samples)

        # Theoretical metrics restricted to [0.01, 0.99]
        worst_cr_m, worst_q_m = calculate_worst_cr_theoretical(t_demands, p_samples)
        worst_cr_b, worst_q_b = calculate_worst_cr_theoretical(t_demands, global_pool)

        r_gpt, r_base = [], []
        for q in Q_VALUES_PLOT:
            a_star, a_gpt, a_base = [get_discrete_quantile_candidates(d, q)[0] for d in [t_demands, p_samples, global_pool]]
            lo, lg, lb = [calculate_newsvendor_loss(a, t_demands, q) for a in [a_star, a_gpt, a_base]]
            rg, rb = [(lo/x if x > 1e-9 else 1.0) for x in [lg, lb]]
            r_gpt.append(rg); r_base.append(rb)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 2), 'CR_gpt': rg, 'CR_base': rb})

        summary_results.append({
            'article_id': art_id, 'method': 'Relevance Selection Method', 'true_mean': t_mean, 'pred_mean': p_mean,
            'abs_error': abs(p_mean - t_mean), 'squared_error': (p_mean - t_mean)**2,
            'AverageCR': np.mean(r_gpt), 'WorstCR': worst_cr_m, 'WorstCase_Q': worst_q_m,
            'Wasserstein': wasserstein_distance(t_demands, p_samples), 'Kolmogorov': ks_2samp(t_demands, p_samples).statistic
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': t_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - t_mean), 'squared_error': (global_mean - t_mean)**2,
            'AverageCR': np.mean(r_base), 'WorstCR': worst_cr_b, 'WorstCase_Q': worst_q_b,
            'Wasserstein': wasserstein_distance(t_demands, global_pool), 'Kolmogorov': ks_2samp(t_demands, global_pool).statistic
        })

    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS WITH MEDIAN AND 95% CI
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*95)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CONFIDENCE INTERVALS)      ")
    print("="*95)

    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'Wasserstein', 'Kolmogorov']
    # Aggregating median along with other stats
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_data = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        print(f"{'Metric':15} | {'Mean':>8} | {'Median':>8} | {'95% CI Range':>20}")
        print("-" * 75)
        m_row = {'Method': method}
        for m in metrics:
            avg, med, std, n = stats_agg.loc[method, (m, 'mean')], stats_agg.loc[method, (m, 'median')], stats_agg.loc[method, (m, 'std')], stats_agg.loc[method, (m, 'count')]
            sem = std / np.sqrt(n)
            margin = 1.96 * sem
            lower, upper = avg - margin, avg + margin
            print(f"{m:15} | {avg:8.4f} | {med:8.4f} | [{lower:6.4f}, {upper:6.4f}]")
            m_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": lower, f"{m}_upper": upper})
        ci_data.append(m_row)

    pd.DataFrame(ci_data).to_csv(f"{PLOT_DIR}/confidence_intervals.csv", index=False)

    sns.set_theme(style="whitegrid")
    
    # Boxplot for Average CR (Visible Median)
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('Mistral Large: Decision Quality (Average CR)\n(Line inside box represents the Median)')
    plt.savefig(f"{PLOT_DIR}/1_boxplot_AverageCR.png")
    plt.close()

    # Violin Plot for Worst-Case Quantile (Failure Analysis)
    plt.figure(figsize=(10, 6))
    sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, palette="Pastel1", inner="quartile")
    plt.title('Distribution of Worst-Case Quantiles (q)\n(Concentration shows the service levels of highest risk)')
    plt.savefig(f"{PLOT_DIR}/2_worst_case_q_dist.png")
    plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CONFIDENCE INTERVALS)      

>>> METHOD: Relevance Selection Method
Metric          |     Mean |   Median |         95% CI Range
---------------------------------------------------------------------------
abs_error       |  34.3447 |  28.6712 | [29.9520, 38.7375]
squared_error   | 2676.4041 | 822.0384 | [1430.8343, 3921.9740]
AverageCR       |   0.6479 |   0.7077 | [0.6218, 0.6740]
WorstCR         |   0.2327 |   0.1706 | [0.2108, 0.2545]
Wasserstein     |  39.2868 |  33.1417 | [35.0629, 43.5107]
Kolmogorov      |   0.4081 |   0.3845 | [0.3871, 0.4291]

>>> METHOD: Baseline (Global)
Metric          |     Mean |   Median |         95% CI Range
---------------------------------------------------------------------------
abs_error       |  34.2066 |  28.5620 | [29.7245, 38.6886]
squa

/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/607871298.py:171: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/607871298.py:178: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='method', y='WorstCase_Q', data=summary_df, palette="Pastel1", inner="quartile")


### Bounded Normal

#### GPT-4o

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import norm, wasserstein_distance, kstest, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'gpt_predictions_random_full_4o_new.csv'   
METRICS_OUT_FILE  = 'final_metrics_summary_descrip_4onew_bound_CI_99.csv'        
DETAILED_OUT_FILE = 'final_metrics_detailed_descrip_4onew_bound_CI_99.csv'       
PLOT_DIR          = 'plots_random_4o_descri_4onew_bound_CI_99'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

# Using a denser linspace for quantile granularity
Q_VALUES = np.linspace(0.001, 1, 2000, endpoint=True)
print(Q_VALUES)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    decision_a = np.array(decision_a)
    demands_xi = np.array(demands_xi)
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    return np.mean(q * underage + (1 - q) * overage)

def get_discrete_quantile(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return 0
    idx = int(np.ceil(n * q)) - 1
    return sorted_data[max(0, min(idx, n - 1))]

def get_normal_quantile(mu, sigma, q, global_max):
    """
    Returns the quantile of a normal distribution bounded between 0 and global_max.
    """
    if pd.isna(mu) or pd.isna(sigma): return 0
    sigma = max(sigma, 0.001)
    # Calculate the theoretical normal quantile
    raw_q = norm.ppf(q, loc=mu, scale=sigma)
    # Clip the result to [0, global_max]
    return np.clip(raw_q, 0, global_max)

# ==========================================
# MAIN LOGIC
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        gpt_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    # Extract global pool and define the maximum boundary
    global_pool = np.sort(true_df[sales_cols].values.flatten()[~np.isnan(true_df[sales_cols].values.flatten())])
    global_mean = np.mean(global_pool)
    GLOBAL_MAX = np.max(global_pool) # The upper bound for our predictions

    summary_results, detailed_results = [], []
    unique_items = gpt_df['article_id'].unique()

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        t_row = true_df[true_df['article_id'] == art_id]
        if t_row.empty: continue
        
        t_demands = np.sort(t_row[sales_cols].values.flatten()[~np.isnan(t_row[sales_cols].values.flatten())])
        t_mean = np.mean(t_demands)

        gpt_row = gpt_df[gpt_df['article_id'] == art_id].iloc[0]
        pred_mean, pred_std = gpt_row.get('pred_mean'), gpt_row.get('pred_std')
        if pd.isna(pred_mean) or pd.isna(pred_std): continue

        # Generate synthetic samples and apply clipping to [0, GLOBAL_MAX]
        raw_samples = np.random.normal(pred_mean, max(pred_std, 0.001), size=5000)
        synth_samples = np.clip(raw_samples, 0, GLOBAL_MAX)
        
        wass_gpt = wasserstein_distance(t_demands, synth_samples)
        wass_base = wasserstein_distance(t_demands, global_pool)
        
        # KS-test compares true demands against the normal distribution parameters
        ks_gpt, _ = kstest(t_demands, 'norm', args=(pred_mean, max(pred_std, 0.001)))
        ks_base, _ = ks_2samp(t_demands, global_pool)

        r_gpt, r_base = [], []
        for q in Q_VALUES:
            a_star = get_discrete_quantile(t_demands, q)
            # Use the bounded quantile function for GPT
            a_gpt = get_normal_quantile(pred_mean, pred_std, q, GLOBAL_MAX)
            a_base = get_discrete_quantile(global_pool, q)
            
            lo, lg, lb = [calculate_newsvendor_loss(a, t_demands, q) for a in [a_star, a_gpt, a_base]]
            rg, rb = [min(lo/x, 1.0) if x > 1e-9 else 1.0 for x in [lg, lb]]
            r_gpt.append(rg); r_base.append(rb)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 4), 'CR_gpt': rg, 'CR_base': rb})

        worst_q_gpt = Q_VALUES[np.argmin(r_gpt)]
        worst_q_base = Q_VALUES[np.argmin(r_base)]

        summary_results.append({
            'article_id': art_id, 'method': 'Relevance Selection Method', 'true_mean': t_mean, 'pred_mean': pred_mean,
            'abs_error': abs(pred_mean - t_mean), 'squared_error': (pred_mean - t_mean)**2,
            'AverageCR': np.mean(r_gpt), 'WorstCR': np.min(r_gpt), 'WorstCR_Quantile': worst_q_gpt,
            'Wasserstein': wass_gpt, 'Kolmogorov': ks_gpt
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': t_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - t_mean), 'squared_error': (global_mean - t_mean)**2,
            'AverageCR': np.mean(r_base), 'WorstCR': np.min(r_base), 'WorstCR_Quantile': worst_q_base,
            'Wasserstein': wass_base, 'Kolmogorov': ks_base
        })

    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS & VISUALIZATION
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*85)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      ")
    print("="*85)

    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'Wasserstein', 'Kolmogorov']
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_list = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        m_row = {'Method': method}
        print(f"{'Metric':15} | {'Mean':>8} | {'Median':>8} | {'95% CI Range':>22}")
        print("-" * 65)

        for m in metrics:
            avg, med, std, n = stats_agg.loc[method, (m, ['mean', 'median', 'std', 'count'])]
            margin = 1.96 * (std / np.sqrt(n))
            
            print(f"{m:15} | {avg:8.4f} | {med:8.4f} | [{avg-margin:7.4f}, {avg+margin:7.4f}]")
            m_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": avg-margin, f"{m}_upper": avg+margin})
        ci_list.append(m_row)

    pd.DataFrame(ci_list).to_csv(f"{PLOT_DIR}/confidence_intervals_detailed.csv", index=False)

    q_performance = detailed_df.groupby('quantile_q')[['CR_gpt', 'CR_base']].mean()
    worst_q_avg = q_performance['CR_gpt'].idxmin()
    worst_val_avg = q_performance['CR_gpt'].min()
    print(f"\nWorst Average Performance for GPT-4o occurs at q = {worst_q_avg:.4f} (CR: {worst_val_avg:.4f})")

    plt.figure(figsize=(8, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('GPT-4o Decision Quality (Average CR)\n(Higher is Better; Line in box is Median)')
    plt.savefig(f"{PLOT_DIR}/1_comparison_boxplot_CR.png")
    plt.close()

    plt.figure(figsize=(8, 6))
    sns.violinplot(x='method', y='WorstCR_Quantile', data=summary_df, palette="Pastel1")
    plt.title('Distribution of Quantiles where Worst CR Occurs\n(Identifying failure points per item)')
    plt.savefig(f"{PLOT_DIR}/2_worst_quantile_distribution.png")
    plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[0.001      0.00149975 0.0019995  ... 0.9990005  0.99950025 1.        ]
[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      

>>> METHOD: Relevance Selection Method
Metric          |     Mean |   Median |           95% CI Range
-----------------------------------------------------------------
abs_error       |  31.1865 |  19.4064 | [26.4343, 35.9388]
squared_error   | 2730.3276 | 376.6143 | [1437.4006, 4023.2545]
AverageCR       |   0.6153 |   0.6808 | [ 0.5861,  0.6444]
WorstCR         |   0.2283 |   0.1504 | [ 0.2045,  0.2521]
Wasserstein     |  33.4705 |  22.6316 | [28.7475, 38.1934]
Kolmogorov      |   0.4910 |   0.4552 | [ 0.4660,  0.5160]

>>> METHOD: Baseline (Global)
Metric          |     Mean |   Median |           95% CI Range
-----------------------------------------------------------------
abs_error       |  

/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/4159738926.py:163: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/4159738926.py:169: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='method', y='WorstCR_Quantile', data=summary_df, palette="Pastel1")


In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import norm, wasserstein_distance, kstest, ks_2samp

# ==========================================
# CONFIGURATION
# ==========================================
TRUE_DATA_FILE    = 'Trousers_new.csv'                         
EXPERIMENT_FILE   = 'gpt_predictions_random_full_4o_new.csv'   
METRICS_OUT_FILE  = 'final_metrics_summary_descrip_4onew_bound2_CI_99.csv'        
DETAILED_OUT_FILE = 'final_metrics_detailed_descrip_4onew_bound2_CI_99.csv'       
PLOT_DIR          = 'plots_random_4o_descri_4onew_bound2_CI_99'

if not os.path.exists(PLOT_DIR):
    os.makedirs(PLOT_DIR)

Q_VALUES = np.linspace(0.01, 0.99, 1000)

# ==========================================
# MATH HELPER FUNCTIONS
# ==========================================
def calculate_newsvendor_loss(decision_a, demands_xi, q):
    decision_a = np.array(decision_a)
    demands_xi = np.array(demands_xi)
    underage = np.maximum(demands_xi - decision_a, 0)
    overage = np.maximum(decision_a - demands_xi, 0)
    return np.mean(q * underage + (1 - q) * overage)

def get_discrete_quantile(sorted_data, q):
    n = len(sorted_data)
    if n == 0: return 0
    idx = int(np.ceil(n * q)) - 1
    return sorted_data[max(0, min(idx, n - 1))]

# Updated to take item_max
def get_normal_quantile(mu, sigma, q, item_max):
    if pd.isna(mu) or pd.isna(sigma): return 0
    sigma = max(sigma, 0.001)
    raw_q = norm.ppf(q, loc=mu, scale=sigma)
    return np.clip(raw_q, 0, item_max)

# ==========================================
# MAIN LOGIC
# ==========================================
def run_evaluation():
    print(f"[INIT] Reading data...")
    try:
        true_df = pd.read_csv(TRUE_DATA_FILE)
        gpt_df = pd.read_csv(EXPERIMENT_FILE)
    except FileNotFoundError as e:
        print(f"[ERROR] {e}"); return None, None

    sales_cols = [c for c in true_df.columns if 'count' in c]
    global_pool = np.sort(true_df[sales_cols].values.flatten()[~np.isnan(true_df[sales_cols].values.flatten())])
    global_mean = np.mean(global_pool)

    summary_results, detailed_results = [], []
    unique_items = gpt_df['article_id'].unique()
    
    total_clips = 0 # To track how often the bound was active

    for i, art_id in enumerate(unique_items):
        if (i+1) % 50 == 0: print(f"Processing item {i+1}...")
        t_row = true_df[true_df['article_id'] == art_id]
        if t_row.empty: continue
        
        t_demands = np.sort(t_row[sales_cols].values.flatten()[~np.isnan(t_row[sales_cols].values.flatten())])
        t_mean = np.mean(t_demands)
        
        # KEY CHANGE: Bounding by ITEM maximum instead of GLOBAL maximum
        item_max = np.max(t_demands)

        gpt_row = gpt_df[gpt_df['article_id'] == art_id].iloc[0]
        pred_mean, pred_std = gpt_row.get('pred_mean'), gpt_row.get('pred_std')
        if pd.isna(pred_mean) or pd.isna(pred_std): continue

        # Generate synthetic samples and apply item-specific clipping
        raw_samples = np.random.normal(pred_mean, max(pred_std, 0.001), size=5000)
        synth_samples = np.clip(raw_samples, 0, item_max)
        
        # Check if clipping actually changed anything for this item
        if np.any(raw_samples > item_max):
            total_clips += 1
        
        wass_gpt = wasserstein_distance(t_demands, synth_samples)
        wass_base = wasserstein_distance(t_demands, global_pool)
        
        ks_gpt, _ = kstest(t_demands, 'norm', args=(pred_mean, max(pred_std, 0.001)))
        ks_base, _ = ks_2samp(t_demands, global_pool)

        r_gpt, r_base = [], []
        for q in Q_VALUES:
            a_star = get_discrete_quantile(t_demands, q)
            # Use the item-specific bound
            a_gpt = get_normal_quantile(pred_mean, pred_std, q, item_max)
            a_base = get_discrete_quantile(global_pool, q)
            
            lo, lg, lb = [calculate_newsvendor_loss(a, t_demands, q) for a in [a_star, a_gpt, a_base]]
            rg, rb = [min(lo/x, 1.0) if x > 1e-9 else 1.0 for x in [lg, lb]]
            r_gpt.append(rg); r_base.append(rb)
            detailed_results.append({'article_id': art_id, 'quantile_q': round(q, 4), 'CR_gpt': rg, 'CR_base': rb})

        summary_results.append({
            'article_id': art_id, 'method': 'Relevance Selection Method', 'true_mean': t_mean, 'pred_mean': pred_mean,
            'abs_error': abs(pred_mean - t_mean), 'squared_error': (pred_mean - t_mean)**2,
            'AverageCR': np.mean(r_gpt), 'WorstCR': np.min(r_gpt), 'WorstCR_Quantile': Q_VALUES[np.argmin(r_gpt)],
            'Wasserstein': wass_gpt, 'Kolmogorov': ks_gpt
        })
        summary_results.append({
            'article_id': art_id, 'method': 'Baseline (Global)', 'true_mean': t_mean, 'pred_mean': global_mean,
            'abs_error': abs(global_mean - t_mean), 'squared_error': (global_mean - t_mean)**2,
            'AverageCR': np.mean(r_base), 'WorstCR': np.min(r_base), 'WorstCR_Quantile': Q_VALUES[np.argmin(r_base)],
            'Wasserstein': wass_base, 'Kolmogorov': ks_base
        })

    print(f"\n[INFO] Clipping was applied to {total_clips} out of {len(unique_items)} items.")
    s_df, d_df = pd.DataFrame(summary_results), pd.DataFrame(detailed_results)
    s_df.to_csv(METRICS_OUT_FILE, index=False)
    d_df.to_csv(DETAILED_OUT_FILE, index=False)
    return s_df, d_df

# ==========================================
# STATISTICS & VISUALIZATION (Identical to previous)
# ==========================================
def generate_stats_and_plots(summary_df, detailed_df):
    if summary_df is None or summary_df.empty: return

    print("\n" + "="*85)
    print("      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      ")
    print("="*85)

    metrics = ['abs_error', 'squared_error', 'AverageCR', 'WorstCR', 'Wasserstein', 'Kolmogorov']
    stats_agg = summary_df.groupby('method')[metrics].agg(['mean', 'median', 'std', 'count'])

    ci_list = []
    for method in summary_df['method'].unique():
        print(f"\n>>> METHOD: {method}")
        m_row = {'Method': method}
        for m in metrics:
            avg, med, std, n = stats_agg.loc[method, (m, ['mean', 'median', 'std', 'count'])]
            margin = 1.96 * (std / np.sqrt(n))
            m_row.update({f"{m}_mean": avg, f"{m}_median": med, f"{m}_lower": avg-margin, f"{m}_upper": avg+margin})
        ci_list.append(m_row)

    pd.DataFrame(ci_list).to_csv(f"{PLOT_DIR}/confidence_intervals_detailed.csv", index=False)

    plt.figure(figsize=(8, 6))
    sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
    plt.title('GPT-4o Decision Quality (Average CR)\n(Bounded by Item Maximum)')
    plt.savefig(f"{PLOT_DIR}/1_comparison_boxplot_CR.png")
    plt.close()

if __name__ == "__main__":
    s_df, d_df = run_evaluation()
    generate_stats_and_plots(s_df, d_df)

[INIT] Reading data...
Processing item 50...
Processing item 100...
Processing item 150...
Processing item 200...
Processing item 250...
Processing item 300...

[INFO] Clipping was applied to 216 out of 300 items.

      PERFORMANCE STATISTICS (MEAN, MEDIAN, & 95% CI)      

>>> METHOD: Relevance Selection Method

>>> METHOD: Baseline (Global)


/var/folders/kk/svkx5ndx1_qcwpl66dw3xyb40000gp/T/ipykernel_15534/3434207815.py:151: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='method', y='AverageCR', data=summary_df, palette="Set2")
